In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

CATALOG = "ml_projects"
SCHEMA = "fraud_project"
NAMESPACE = f"{CATALOG}.{SCHEMA}"

applications_table = f"{NAMESPACE}.applications"
labels_table = f"{NAMESPACE}.labels"
predictions_table = f"{NAMESPACE}.predictions"
monitoring_table = f"{NAMESPACE}.monitoring_results"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {monitoring_table} (
    month_end DATE,
    checked_at TIMESTAMP,
    prediction_count BIGINT,
    labeled_prediction_count BIGINT,
    avg_requested_amount DOUBLE,
    missing_income_rate DOUBLE,
    review_rate DOUBLE,
    precision DOUBLE,
    recall DOUBLE,
    drift_detected BOOLEAN,
    model_version STRING
) USING DELTA
""")

def monitor_one_month(as_of_text):
    as_of = datetime.fromisoformat(as_of_text)

    apps = spark.table(applications_table)
    preds = spark.table(predictions_table)
    labels = (
        spark.table(labels_table)
        .filter(F.col("label_available_at") <= F.lit(as_of))
    )

    # Берём заявки, поступившие в этот месяц.
    month_start = as_of.replace(day=1)

    month_apps = apps.filter(
        (F.col("application_timestamp") >= F.lit(month_start))
        & (F.col("application_timestamp") <= F.lit(as_of))
    )

    month_preds = preds.filter(
        (F.col("prediction_timestamp") >= F.lit(month_start))
        & (F.col("prediction_timestamp") <= F.lit(as_of))
    )

    joined = (
        month_preds.alias("p")
        .join(month_apps.alias("a"), on="application_id", how="left")
        .join(labels.alias("l"), on="application_id", how="left")
    )

    values = joined.agg(
        F.count("*").alias("prediction_count"),
        F.count("fraud_label").alias("labeled_prediction_count"),
        F.avg("requested_amount").alias("avg_requested_amount"),
        F.avg(
            F.when(F.col("verified_monthly_income").isNull(), 1.0)
             .otherwise(0.0)
        ).alias("missing_income_rate"),
        F.avg(F.col("review_required").cast("double")).alias("review_rate"),
        F.sum(
            F.when(
                (F.col("review_required") == True) & (F.col("fraud_label") == 1),
                1
            ).otherwise(0)
        ).alias("true_positives"),
        F.sum(
            F.when(
                (F.col("review_required") == True) & F.col("fraud_label").isNotNull(),
                1
            ).otherwise(0)
        ).alias("reviewed_labeled"),
        F.sum(
            F.when(F.col("fraud_label") == 1, 1).otherwise(0)
        ).alias("actual_fraud"),
    ).first()

    # Сравниваем сумму и пропуски с исходным BASE-набором.
    baseline = (
        apps.filter(F.col("application_id").startswith("BASE-"))
        .agg(
            F.avg("requested_amount").alias("avg_amount"),
            F.avg(
                F.when(F.col("verified_monthly_income").isNull(), 1.0)
                 .otherwise(0.0)
            ).alias("missing_rate"),
        )
        .first()
    )

    tp = values["true_positives"] or 0
    reviewed_labeled = values["reviewed_labeled"] or 0
    actual_fraud = values["actual_fraud"] or 0

    precision_value = tp / reviewed_labeled if reviewed_labeled else None
    recall_value = tp / actual_fraud if actual_fraud else None

    avg_amount = values["avg_requested_amount"]
    missing_rate = values["missing_income_rate"]
    review_rate = values["review_rate"]

    drift = (
        (avg_amount is not None and baseline["avg_amount"] is not None
         and avg_amount > baseline["avg_amount"] * 1.5)
        or
        (missing_rate is not None and baseline["missing_rate"] is not None
         and missing_rate > baseline["missing_rate"] + 0.10)
        or
        (review_rate is not None and review_rate > 0.20)
    )

    version = (
        month_preds.agg(F.max("model_version")).first()[0]
        or "unknown"
    )

    result = [(
        as_of.date(),
        datetime.now(timezone.utc).replace(tzinfo=None),
        values["prediction_count"] or 0,
        values["labeled_prediction_count"] or 0,
        avg_amount,
        missing_rate,
        review_rate,
        precision_value,
        recall_value,
        bool(drift),
        str(version),
    )]

    result_df = spark.createDataFrame(
        result,
        schema="""
            month_end DATE,
            checked_at TIMESTAMP,
            prediction_count LONG,
            labeled_prediction_count LONG,
            avg_requested_amount DOUBLE,
            missing_income_rate DOUBLE,
            review_rate DOUBLE,
            precision DOUBLE,
            recall DOUBLE,
            drift_detected BOOLEAN,
            model_version STRING
        """
    )

    result_df.write.mode("append").saveAsTable(monitoring_table)

In [0]:
import pandas as pd

for month_end in pd.date_range("2025-01-31", "2025-12-31", freq="ME"):
    monitor_one_month(
        month_end.strftime("%Y-%m-%dT23:59:59")
    )

display(
    spark.table("ml_projects.fraud_project.monitoring_results")
    .orderBy("month_end")
)

In [0]:
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import functions as F

NAMESPACE = "ml_projects.fraud_project"

apps = spark.table(f"{NAMESPACE}.applications")
preds = spark.table(f"{NAMESPACE}.predictions")
labels = spark.table(f"{NAMESPACE}.labels")

# Средние показатели исходного набора для сравнения с последующими месяцами
baseline = (
    apps.filter(F.col("application_id").startswith("BASE-"))
    .agg(
        F.avg("requested_amount").alias("avg_amount"),
        F.avg(
            F.when(F.col("verified_monthly_income").isNull(), 1.0)
             .otherwise(0.0)
        ).alias("missing_rate"),
    )
    .first()
)

monitor_rows = []

for month in range(1, 13):
    month_start = pd.Timestamp(2025, month, 1)
    next_month_start = month_start + pd.offsets.MonthBegin(1)
    month_end = next_month_start - pd.Timedelta(seconds=1)

    # Отбираем заявки по дате поступления.
    # Так мы не теряем predictions, записанные в первое число следующего месяца.
    month_apps = (
        apps.filter(
            (F.col("application_timestamp") >= F.lit(month_start.to_pydatetime()))
            & (F.col("application_timestamp") < F.lit(next_month_start.to_pydatetime()))
        )
        .select(
            "application_id",
            "requested_amount",
            "verified_monthly_income",
        )
    )

    month_labels = labels.filter(
        F.col("label_available_at") <= F.lit(month_end.to_pydatetime())
    )

    month_joined = (
        preds
        .join(month_apps, on="application_id", how="inner")
        .join(month_labels, on="application_id", how="left")
    )

    stats = month_joined.agg(
        F.count("*").alias("prediction_count"),
        F.count("fraud_label").alias("labeled_prediction_count"),
        F.avg("requested_amount").alias("avg_requested_amount"),
        F.avg(
            F.when(F.col("verified_monthly_income").isNull(), 1.0)
             .otherwise(0.0)
        ).alias("missing_income_rate"),
        F.avg(F.col("review_required").cast("double")).alias("review_rate"),
        F.sum(
            F.when(
                (F.col("review_required") == True) & (F.col("fraud_label") == 1),
                1
            ).otherwise(0)
        ).alias("true_positives"),
        F.sum(
            F.when(
                (F.col("review_required") == True)
                & F.col("fraud_label").isNotNull(),
                1
            ).otherwise(0)
        ).alias("reviewed_labeled"),
        F.sum(
            F.when(F.col("fraud_label") == 1, 1).otherwise(0)
        ).alias("actual_fraud"),
        F.max("model_version").alias("model_version"),
    ).first()

    true_positives = stats["true_positives"] or 0
    reviewed_labeled = stats["reviewed_labeled"] or 0
    actual_fraud = stats["actual_fraud"] or 0

    precision_value = (
        true_positives / reviewed_labeled if reviewed_labeled else None
    )
    recall_value = true_positives / actual_fraud if actual_fraud else None

    amount_drift = (
        stats["avg_requested_amount"] is not None
        and baseline["avg_amount"] is not None
        and stats["avg_requested_amount"] > baseline["avg_amount"] * 1.5
    )
    missingness_drift = (
        stats["missing_income_rate"] is not None
        and baseline["missing_rate"] is not None
        and stats["missing_income_rate"] > baseline["missing_rate"] + 0.10
    )
    review_rate_drift = (
        stats["review_rate"] is not None
        and stats["review_rate"] > 0.20
    )

    monitor_rows.append((
        month_end.date(),
        datetime.now(timezone.utc).replace(tzinfo=None),
        stats["prediction_count"] or 0,
        stats["labeled_prediction_count"] or 0,
        stats["avg_requested_amount"],
        stats["missing_income_rate"],
        stats["review_rate"],
        precision_value,
        recall_value,
        bool(amount_drift or missingness_drift or review_rate_drift),
        str(stats["model_version"] or "unknown"),
    ))

monitor_schema = """
    month_end DATE,
    checked_at TIMESTAMP,
    prediction_count LONG,
    labeled_prediction_count LONG,
    avg_requested_amount DOUBLE,
    missing_income_rate DOUBLE,
    review_rate DOUBLE,
    precision DOUBLE,
    recall DOUBLE,
    drift_detected BOOLEAN,
    model_version STRING
"""

(
    spark.createDataFrame(monitor_rows, schema=monitor_schema)
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{NAMESPACE}.monitoring_results")
)

display(
    spark.table(f"{NAMESPACE}.monitoring_results")
    .orderBy("month_end")
)

In [0]:
%sql
SELECT
  COUNT(*) AS months,
  SUM(prediction_count) AS predictions,
  SUM(labeled_prediction_count) AS labeled_predictions
FROM ml_projects.fraud_project.monitoring_results;